#  India Marriage Dashboard
### An interactive dashboard to explore Love vs Arranged marriages in India

In [1]:
import numpy as np
import pandas as pd
import panel as pn
pn.extension('tabulator')
import hvplot.pandas

In [2]:

df = pd.read_csv('marriage_data_india.csv')
df['Year_of_Marriage'] = 2024 - df['Years_Since_Marriage']
df['Parental_Approval_Num'] = df['Parental_Approval'].map({'No': 0, 'Partial': 1, 'Yes': 2})
df['Marital_Satisfaction_Num'] = df['Marital_Satisfaction'].map({'Low': 0, 'Medium': 1, 'High': 2})
df['Divorce_Num'] = df['Divorce_Status'].map({'No': 0, 'Yes': 1})


In [3]:
df['Education_Num'] = df['Education_Level'].map({'School': 0, 'Graduate': 1, 'Postgraduate': 2, 'PhD': 3})
df['Income_Num'] = df['Income_Level'].map({'Low': 0, 'Middle': 1, 'High': 2})
df['Inter_Caste_Num'] = df['Inter-Caste'].map({'No': 0, 'Yes': 1})
df['Inter_Religion_Num'] = df['Inter-Religion'].map({'No': 0, 'Yes': 1})
df['Dowry_Num'] = df['Dowry_Exchanged'].map({'No': 0, 'Not Disclosed': 1, 'Yes': 2})
print('Data loaded successfully! Total rows:', len(df))
print('Year range:', df['Year_of_Marriage'].min(), 'to', df['Year_of_Marriage'].max())
df.head(3)

Data loaded successfully! Total rows: 10000
Year range: 1975 to 2023


,ID,Marriage_Type,Age_at_Marriage,Gender,Education_Level,Caste_Match,Religion,Parental_Approval,Urban_Rural,Dowry_Exchanged,...,Inter-Religion,Year_of_Marriage,Parental_Approval_Num,Marital_Satisfaction_Num,Divorce_Num,Education_Num,Income_Num,Inter_Caste_Num,Inter_Religion_Num,Dowry_Num
0,1,Love,23,Male,Graduate,Different,Hindu,No,Urban,No,...,No,1990,0,1,1,1,1,0,0,0
1,2,Love,28,Female,School,Same,Hindu,Yes,Rural,Yes,...,Yes,1982,2,0,0,0,1,0,1,2
2,3,Arranged,39,Male,Postgraduate,Same,Muslim,Yes,Rural,No,...,No,1999,2,1,0,2,2,0,0,0


In [4]:
year_slider = pn.widgets.IntSlider(
    name='Year of Marriage (Up to)',
    start=1975,
    end=2023,
    step=1,
    value=2010
)

In [5]:
yaxis_trend = pn.widgets.RadioButtonGroup(
    name='Show trend for',
    options=['Love', 'Arranged'],
    button_type='primary',
    button_style='outline'
)

In [6]:
yaxis_metric = pn.widgets.RadioButtonGroup(
    name='Compare by',
    options=['Divorce_Num', 'Parental_Approval_Num', 'Marital_Satisfaction_Num'],
    button_type='success',
    button_style='outline'
)

In [7]:
yaxis_cross = pn.widgets.RadioButtonGroup(
    name='Cross Marriage Type',
    options=['Inter_Caste_Num', 'Inter_Religion_Num'],
    button_type='warning',
    button_style='outline'
)

year_slider

IntSlider(end=2023, name='Year of Marriage (..., start=1975, value=2010)

In [8]:
import hvplot.pandas  
def make_trend_plot(year):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby(['Year_of_Marriage', 'Marriage_Type'])['ID'].count()
        .reset_index()
        .rename(columns={'ID': 'Number_of_Marriages'})
        .sort_values('Year_of_Marriage')
    )
    return data.hvplot(
        x='Year_of_Marriage', y='Number_of_Marriages', by='Marriage_Type',
        line_width=2.5, title='Love vs Arranged Marriages Over the Years',
        xlabel='Year of Marriage', ylabel='Number of Marriages',
        color=['#e74c3c', '#3498db'], responsive=True, min_height=380
    )
trend_panel = pn.bind(make_trend_plot, year=year_slider)
def make_trend_table(year):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby(['Year_of_Marriage', 'Marriage_Type'])['ID'].count()
        .reset_index()
        .rename(columns={'ID': 'Number_of_Marriages'})
        .sort_values('Year_of_Marriage')
    )
    return pn.widgets.Tabulator(data, pagination='remote', page_size=12, sizing_mode='stretch_width')
trend_table_panel = pn.bind(make_trend_table, year=year_slider)

def make_approval_plot(year):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby(['Marriage_Type', 'Parental_Approval'])['ID'].count()
        .reset_index()
        .rename(columns={'ID': 'Count'})
        .sort_values('Marriage_Type')
    )
    return data.hvplot(
        kind='bar', x='Parental_Approval', y='Count', by='Marriage_Type',
        title='Parental Approval: Love vs Arranged', xlabel='Parental Approval',
        ylabel='Number of Couples', color=['#e74c3c', '#3498db'],
        rot=0, responsive=True, min_height=380
    )

approval_panel = pn.bind(make_approval_plot, year=year_slider)

def make_cross_plot(year, cross_col):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby('Marriage_Type')[cross_col].mean()
        .reset_index()
    )
    return data.hvplot(
        kind='bar', x='Marriage_Type', y=cross_col,
        title='Inter-Caste / Inter-Religion Marriage Rate',
        xlabel='Marriage Type', ylabel='Rate (0=None, 1=All)',
        color=['#9b59b6', '#1abc9c'], responsive=True, min_height=380
    )

cross_panel = pn.bind(make_cross_plot, year=year_slider, cross_col=yaxis_cross)

def make_divorce_plot(year):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby('Marriage_Type')['Divorce_Num'].mean()
        .reset_index()
        .rename(columns={'Divorce_Num': 'Divorce_Rate'})
    )
    return data.hvplot(
        kind='bar', x='Marriage_Type', y='Divorce_Rate',
        title='Divorce Rate: Love vs Arranged Marriages',
        xlabel='Marriage Type', ylabel='Divorce Rate',
        color=['#e74c3c', '#3498db'], ylim=(0, 0.3),
        responsive=True, min_height=380
    )

divorce_panel = pn.bind(make_divorce_plot, year=year_slider)

def make_satisfaction_plot(year):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby(['Marriage_Type', 'Marital_Satisfaction'])['ID'].count()
        .reset_index()
        .rename(columns={'ID': 'Count'})
        .sort_values('Marriage_Type')
    )
    return data.hvplot(
        kind='bar', x='Marital_Satisfaction', y='Count', by='Marriage_Type',
        title='Marital Satisfaction: Love vs Arranged',
        xlabel='Satisfaction Level', ylabel='Number of Couples',
        color=['#e74c3c', '#3498db'], responsive=True, min_height=380
    )

satisfaction_panel = pn.bind(make_satisfaction_plot, year=year_slider)

def make_edu_plot(year):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby(['Education_Level', 'Marriage_Type'])['ID'].count()
        .reset_index()
        .rename(columns={'ID': 'Count'})
        .sort_values('Education_Level')
    )
    return data.hvplot(
        kind='bar', x='Education_Level', y='Count', by='Marriage_Type',
        title='Education Level by Marriage Type',
        xlabel='Education Level', ylabel='Number of People',
        color=['#e74c3c', '#3498db'], responsive=True, min_height=380
    )

edu_panel = pn.bind(make_edu_plot, year=year_slider)

def make_urban_plot(year):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby(['Urban_Rural', 'Marriage_Type'])['ID'].count()
        .reset_index()
        .rename(columns={'ID': 'Count'})
    )
    return data.hvplot(
        kind='bar', x='Urban_Rural', y='Count', by='Marriage_Type',
        title='Urban vs Rural: Love or Arranged?',
        xlabel='Area', ylabel='Number of Couples',
        color=['#e74c3c', '#3498db'], responsive=True, min_height=380
    )

urban_panel = pn.bind(make_urban_plot, year=year_slider)

def make_dowry_plot(year):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby(['Marriage_Type', 'Dowry_Exchanged'])['ID'].count()
        .reset_index()
        .rename(columns={'ID': 'Count'})
    )
    return data.hvplot(
        kind='bar', x='Dowry_Exchanged', y='Count', by='Marriage_Type',
        title='Dowry Exchange: Love vs Arranged Marriages',
        xlabel='Dowry Exchanged', ylabel='Number of Couples',
        color=['#e74c3c', '#3498db'], responsive=True, min_height=380
    )

dowry_panel = pn.bind(make_dowry_plot, year=year_slider)

def make_age_plot(year):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby(['Year_of_Marriage', 'Marriage_Type'])['Age_at_Marriage'].mean()
        .reset_index()
        .rename(columns={'Age_at_Marriage': 'Avg_Age'})
        .sort_values('Year_of_Marriage')
    )
    return data.hvplot(
        x='Year_of_Marriage', y='Avg_Age', by='Marriage_Type',
        line_width=2, title='Average Age at Marriage Over the Years',
        xlabel='Year of Marriage', ylabel='Average Age',
        color=['#e74c3c', '#3498db'], responsive=True, min_height=380
    )

age_panel = pn.bind(make_age_plot, year=year_slider)

def make_income_plot(year):
    data = (
        df[df.Year_of_Marriage <= year]
        .groupby(['Income_Level', 'Marriage_Type'])['ID'].count()
        .reset_index()
        .rename(columns={'ID': 'Count'})
    )
    return data.hvplot(
        kind='bar', x='Income_Level', y='Count', by='Marriage_Type',
        title='Income Level by Marriage Type',
        xlabel='Income Level', ylabel='Number of Couples',
        color=['#e74c3c', '#3498db'], responsive=True, min_height=380
    )
income_panel = pn.bind(make_income_plot, year=year_slider)


## Final Dashboard Layout

In [9]:
template = pn.template.FastListTemplate(
    title=' India Marriage Dashboard',

    sidebar=[
        pn.pane.Markdown("# India Marriage Data"),
        pn.pane.Markdown(
            """#### This dashboard explores **10,000 marriages** in India.\n\n
Compare **Love** vs **Arranged** marriages across:\n
- Trends over time\n
- Parental approval\n
- Divorce rates\n
- Satisfaction levels\n
- Inter-caste & religion\n
- Education, Income, Dowry\n"""
        ),
        pn.pane.Markdown("## ⚙️ Filter Settings"),
        year_slider,
        pn.pane.Markdown("---"),
        pn.pane.Markdown("**Inter-Caste / Religion Toggle:**"),
        yaxis_cross,
    ],

    main=[
        pn.Row(
            pn.Column(pn.pane.Markdown("### 📈 Marriage Trends Over Time"), trend_panel, sizing_mode='stretch_width'),
        ),
        pn.Row(
            pn.Column(pn.pane.Markdown("### \u200d\u200d Parental Approval by Marriage Type"), approval_panel),
            pn.Column(pn.pane.Markdown("###  Inter-Caste / Inter-Religion Marriages"), cross_panel),
        ),
        pn.Row(
            pn.Column(pn.pane.Markdown("###  Divorce Rate Comparison"), divorce_panel),
            pn.Column(pn.pane.Markdown("###  Marital Satisfaction Levels"), satisfaction_panel),
        ),
        pn.Row(
            pn.Column(pn.pane.Markdown("###  Average Age at Marriage Over Time"), age_panel),
            pn.Column(pn.pane.Markdown("###  Dowry Exchange by Marriage Type"), dowry_panel),
        ),
        pn.Row(
            pn.Column(pn.pane.Markdown("###  Education Level by Marriage Type"), edu_panel),
            pn.Column(pn.pane.Markdown("###  Urban vs Rural: Who Chooses What?"), urban_panel),
        ),
        pn.Row(
            pn.Column(pn.pane.Markdown("###  Income Level by Marriage Type"), income_panel),
        ),
        pn.Row(
            pn.Column(pn.pane.Markdown("### 📋 Raw Data Table (Marriage Trends)"), trend_table_panel, sizing_mode='stretch_width'),
        ),
    ],

    accent_base_color='#e74c3c',
    header_background='#c0392b',
    background_color='#fff5f5',
)

template.servable()
template.show()


Launching server at http://localhost:56354
